# Lab 1b · Tu primer agente con Microsoft Agent Framework

El mismo agente del Lab 1a… en 10 líneas. Después le daremos un **rol**, **herramientas propias**, **memoria**,
**salida estructurada** y lo veremos por dentro en **DevUI**. Tiempo estimado: 25 minutos.

In [ ]:
import json
from typing import Annotated

from agent_framework import Agent, tool

import almansa
from almansa import negocio

cliente = almansa.cliente()  # cliente de Gemini con límite de llamadas, reintentos y contador ya integrados

## Paso 1 · Un agente en 10 líneas

In [ ]:
agente = Agent(
    client=cliente,
    name="analista",
    instructions=(
        "Eres el analista de datos de Almansa Market (cadena ficticia). Usa las herramientas y no inventes datos. "
        "Hay datos del 2025-01-01 al 2026-10-31. Responde en español, breve y con cifras."
    ),
    tools=[negocio.ventas_por_categoria, negocio.ventas_por_tienda, negocio.top_productos],
)

respuesta = await agente.run(
    "¿Qué categoría vendió más en el tercer trimestre de 2026 y qué tienda vendió más de esa categoría? "
    "Compáralo con el tercer trimestre de 2025."
)
print(respuesta.text)

In [ ]:
almansa.mostrar_traza(respuesta)  # los pasos que ha dado el agente por dentro

**¿Qué ha hecho el framework por ti?** El bucle, la ejecución de herramientas (también en paralelo), los errores,
las firmas de razonamiento… y el **JSON Schema**, que genera a partir de los *type hints* y del *docstring*:

In [ ]:
print(json.dumps(tool(negocio.top_productos).to_json_schema_spec(), indent=2, ensure_ascii=False))

## Paso 2 · Las instrucciones definen el "puesto de trabajo" ✏️

Compara el mismo agente con dos instrucciones distintas. Después escribe las de tu equipo
(Finanzas, Marketing, Operaciones y Clientes, o Documentación y Compliance).

In [ ]:
pregunta = "¿Cómo fueron las ventas de octubre de 2026 frente a octubre de 2025?"
for estilo in ["Responde en una sola frase.", "Responde como un informe para el comité de dirección: 3 viñetas y una recomendación."]:
    agente_estilo = Agent(client=cliente, name="analista", instructions=f"Eres el analista de Almansa Market. Usa las herramientas. {estilo}",
                          tools=[negocio.ventas_por_categoria, negocio.ventas_mensuales])
    print(f"--- {estilo}\n{(await agente_estilo.run(pregunta)).text}\n")

In [ ]:
INSTRUCCIONES_EQUIPO = """✏️ TODO: describe el rol del agente de tu equipo:
- ¿Quién es y qué le preocupa?
- ¿Qué debe hacer siempre (verificar con datos, citar fuentes...)? ¿Qué no debe hacer nunca?
- ¿Qué formato y longitud de respuesta esperas?"""

## Paso 3 · Crea tu propia herramienta ✏️

Una herramienta es una función con **type hints**, descripciones en `Annotated` y un **docstring** claro.
Mira las tablas disponibles con `print(almansa.esquema())`. Ideas por equipo:

| Equipo | Herramienta |
|---|---|
| Finanzas | margen bruto por categoría o por tienda |
| Marketing | clientes por segmento, ventas por canal (tienda/web/app) |
| Operaciones y Clientes | tickets por categoría y tienda, tiempos de resolución |
| Documentación y Compliance | clientes con/sin consentimiento de marketing, reseñas por producto |

In [ ]:
@tool
def mi_herramienta(
    desde: Annotated[str, "Fecha inicial AAAA-MM-DD"],
    hasta: Annotated[str, "Fecha final AAAA-MM-DD"],
) -> str:
    """✏️ TODO: describe qué devuelve. El modelo decide usarla leyendo SOLO esto."""
    df = almansa.consultar(
        "SELECT ... FROM v_ventas_lineas WHERE estado = 'completada' AND fecha BETWEEN ? AND ?",  # ✏️ TODO
        (desde, hasta),
    )
    return df.to_csv(index=False)

MI_HERRAMIENTA = mi_herramienta
print(MI_HERRAMIENTA.func("2026-07-01", "2026-09-30"))  # pruébala antes de dársela al agente

In [ ]:
agente_equipo = Agent(
    client=cliente,
    name="agente_equipo",
    instructions=INSTRUCCIONES_EQUIPO,
    tools=[MI_HERRAMIENTA, negocio.ventas_por_categoria, negocio.ventas_por_tienda, negocio.objetivos_vs_real],
)
respuesta = await agente_equipo.run("✏️ TODO: una pregunta de tu área que obligue a usar tu herramienta")
print(respuesta.text)
almansa.mostrar_traza(respuesta)

## Paso 4 · Memoria de conversación (sesiones)

Por defecto cada `run` empieza de cero. Con una **sesión**, el agente recuerda lo hablado.

In [ ]:
sesion = agente.create_session()
r1 = await agente.run("¿Cuánto vendió la tienda de Valencia (T04) en el primer trimestre de 2026?", session=sesion)
print(r1.text, "\n")
r2 = await agente.run("¿Y en el mismo trimestre de 2025? ¿A qué crees que se debe la diferencia?", session=sesion)
print(r2.text)

🤔 El agente solo ve datos de ventas: para explicar el *porqué* puede que especule. ¿Dónde estaría la respuesta?
(Pista: en los documentos internos de Almansa… lo resolveremos en la **sesión 2** con RAG.)

## Paso 5 · Salida estructurada

Si la respuesta la va a consumir otro sistema (un Excel, un cuadro de mando, otro agente…), pide un **esquema**.

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field


class CategoriaTrimestre(BaseModel):
    categoria: str
    ventas_eur: float
    variacion_pct: float = Field(description="Variación porcentual frente al mismo trimestre del año anterior")


class InformeTrimestral(BaseModel):
    trimestre: str
    categorias: list[CategoriaTrimestre]
    conclusion: str = Field(description="Una frase para la dirección")


r = await agente.run("Prepara el informe del T3 de 2026 por categoría comparado con el T3 de 2025.",
                     options={"response_format": InformeTrimestral})
informe = r.value
print(informe.conclusion)
pd.DataFrame([c.model_dump() for c in informe.categorias])

## Paso 6 · Mira dentro del agente con DevUI

1. Abre un **terminal** (menú ☰ → Terminal → New Terminal) y ejecuta:
   ```bash
   python sesion1/app_devui.py
   ```
2. Abre el puerto **8080** (pestaña **PUERTOS** → 🌐).
3. Pregunta: *"Las ventas de Hogar en Andalucía han caído en el T3 de 2026. ¿Qué ha pasado?"*
4. Observa la traza: ¿qué herramientas usa?, ¿en qué orden?, ¿cuánto tarda?, ¿cuántos tokens?

Para parar el servidor: `Ctrl + C` en el terminal.

## 🧩 Reto de equipo: Agent Canvas

En el cierre de la sesión empezaréis en equipo el **Agent Canvas** ([docs/agent_canvas.md](../docs/agent_canvas.md))
de un agente para vuestra área: 5 minutos en clase para los bloques 1, 3, 4 y 7 (problema, herramientas,
autonomía y coste y valor) y el resto, en casa.
En la sesión 2 construiréis ese agente conectado a los datos y a los documentos de Almansa.

In [ ]:
almansa.uso()